In [1]:
from google.colab import files
import pandas as pd
import numpy as np

uploaded = files.upload()

Saving 국민기초생활 수급자 동별 현황(202405).xlsx to 국민기초생활 수급자 동별 현황(202405).xlsx


In [3]:
file_path = "국민기초생활 수급자 동별 현황(202405).xlsx"

xls = pd.ExcelFile(file_path)

print("시트 목록:")
print(xls.sheet_names)

시트 목록:
['기초생활보장 수급권자 동별 현황']


In [4]:
sheet_name = xls.sheet_names[0]

raw = pd.read_excel(
    file_path,
    sheet_name=sheet_name,
    header=None
)

print("원본 크기:", raw.shape)

display(raw.head(15))

원본 크기: (4438, 5)


,0,1,2,3,4
0,NaN,NaN,NaN,NaN,NaN
1,읍면동,자격,연령구간,NaN,수급권자수
2,NaN,NaN,NaN,NaN,NaN
3,종로구,기초생계급여,18세미만,NaN,44
4,NaN,NaN,18~64세,NaN,312
5,NaN,NaN,65세이상,NaN,140
6,NaN,기초의료급여,18~64세,NaN,2
7,NaN,NaN,65세이상,NaN,2
8,청운효자동,기초생계급여,18세미만,NaN,10
9,NaN,NaN,18~64세,NaN,59


In [5]:
welfare_raw = pd.read_excel(
    file_path,
    sheet_name=sheet_name,
    header=1
)

print("컬럼:")
print(welfare_raw.columns.tolist())

print("\n크기:", welfare_raw.shape)

display(welfare_raw.head(15))

컬럼:
['읍면동', '자격', '연령구간', 'Unnamed: 3', '수급권자수']

크기: (4436, 5)


,읍면동,자격,연령구간,Unnamed: 3,수급권자수
0,NaN,NaN,NaN,NaN,NaN
1,종로구,기초생계급여,18세미만,NaN,44.0
2,NaN,NaN,18~64세,NaN,312.0
3,NaN,NaN,65세이상,NaN,140.0
4,NaN,기초의료급여,18~64세,NaN,2.0
5,NaN,NaN,65세이상,NaN,2.0
6,청운효자동,기초생계급여,18세미만,NaN,10.0
7,NaN,NaN,18~64세,NaN,59.0
8,NaN,NaN,65세이상,NaN,72.0
9,NaN,기초의료급여,18세미만,NaN,1.0


In [6]:
gangnam_rows = welfare_raw[
    welfare_raw["읍면동"].astype(str).str.contains("강남구", na=False)
]

print("강남구가 직접 기록된 행:")
display(gangnam_rows)

강남구가 직접 기록된 행:


,읍면동,자격,연령구간,Unnamed: 3,수급권자수
3786,강남구,기초생계급여,18세미만,NaN,21.0


In [7]:
welfare = welfare_raw[
    ["읍면동", "자격", "연령구간", "수급권자수"]
].copy()

# 원본 읍면동 값 보존
welfare["읍면동_원본"] = welfare["읍면동"]

# 병합셀로 인해 생긴 빈 값 복원
welfare["읍면동"] = welfare["읍면동"].ffill()
welfare["자격"] = welfare["자격"].ffill()

print("전처리 후 크기:", welfare.shape)

display(welfare.iloc[3786:3810])

전처리 후 크기: (4436, 5)


,읍면동,자격,연령구간,수급권자수,읍면동_원본
3786,강남구,기초생계급여,18세미만,21.0,강남구
3787,강남구,기초생계급여,18~64세,249.0,NaN
3788,강남구,기초생계급여,65세이상,198.0,NaN
3789,강남구,기초의료급여,18~64세,2.0,NaN
3790,강남구,기초의료급여,65세이상,7.0,NaN
3791,강남구,기초주거급여,18~64세,1.0,NaN
3792,강남구,기초주거급여,65세이상,1.0,NaN
3793,신사동,기초생계급여,18세미만,4.0,신사동
3794,신사동,기초생계급여,18~64세,62.0,NaN
3795,신사동,기초생계급여,65세이상,53.0,NaN


In [8]:
gangnam_start = 3786

next_area_names = (
    welfare_raw.loc[gangnam_start:, "읍면동"]
    .dropna()
    .head(30)
)

print(next_area_names.to_string())

3786     강남구
3793     신사동
3803    논현1동
3812    논현2동
3822    압구정동
3831     청담동
3840    삼성1동
3850    삼성2동
3860    대치1동
3865    대치2동
3874    대치4동
3885    역삼1동
3896    역삼2동
3906    도곡1동
3912    도곡2동
3921    개포1동
3929    개포2동
3938    개포3동
3948    개포4동
3958     세곡동
3969    일원본동
3976    일원1동
3987     수서동
3997     송파구
4003    풍납1동
4013    풍납2동
4023    거여1동
4033    거여2동
4043    마천1동
4053    마천2동


In [9]:
gangnam_welfare = welfare.loc[3793:3996].copy()

# 분석에 필요 없는 원본 확인용 컬럼 제거
gangnam_welfare = gangnam_welfare[
    ["읍면동", "자격", "연령구간", "수급권자수"]
].copy()

print("강남구 데이터 크기:", gangnam_welfare.shape)
print("행정동 수:", gangnam_welfare["읍면동"].nunique())

print("\n행정동 목록:")
print(gangnam_welfare["읍면동"].unique())

print("\n자격 종류:")
print(gangnam_welfare["자격"].unique())

print("\n연령구간:")
print(gangnam_welfare["연령구간"].unique())

print("\n결측치:")
print(gangnam_welfare.isna().sum())

강남구 데이터 크기: (204, 4)
행정동 수: 22

행정동 목록:
['신사동' '논현1동' '논현2동' '압구정동' '청담동' '삼성1동' '삼성2동' '대치1동' '대치2동' '대치4동'
 '역삼1동' '역삼2동' '도곡1동' '도곡2동' '개포1동' '개포2동' '개포3동' '개포4동' '세곡동' '일원본동'
 '일원1동' '수서동']

자격 종류:
['기초생계급여' '기초의료급여' '기초주거급여' '기초교육급여']

연령구간:
['18세미만' '18~64세' '65세이상']

결측치:
읍면동      0
자격       0
연령구간     0
수급권자수    0
dtype: int64


In [10]:
dong_check = (
    gangnam_welfare
    .groupby("읍면동")
    .size()
    .reset_index(name="행수")
)

display(dong_check)

,읍면동,행수
0,개포1동,8
1,개포2동,9
2,개포3동,10
3,개포4동,10
4,논현1동,9
5,논현2동,10
6,대치1동,5
7,대치2동,9
8,대치4동,11
9,도곡1동,6


In [11]:
combination_check = (
    gangnam_welfare
    .groupby(["자격", "연령구간"])
    .agg(
        dong_count=("읍면동", "nunique"),
        total_recipients=("수급권자수", "sum")
    )
    .reset_index()
)

display(combination_check)

,자격,연령구간,dong_count,total_recipients
0,기초교육급여,18~64세,8,12.0
1,기초교육급여,18세미만,18,79.0
2,기초생계급여,18~64세,22,5102.0
3,기초생계급여,18세미만,20,594.0
4,기초생계급여,65세이상,22,5173.0
5,기초의료급여,18~64세,18,401.0
6,기초의료급여,18세미만,11,27.0
7,기초의료급여,65세이상,21,311.0
8,기초주거급여,18~64세,22,2261.0
9,기초주거급여,18세미만,20,444.0


In [12]:
import itertools

dongs = gangnam_welfare["읍면동"].unique()
qualifications = gangnam_welfare["자격"].unique()
age_groups = ["18세미만", "18~64세", "65세이상"]

full_index = pd.MultiIndex.from_product(
    [dongs, qualifications, age_groups],
    names=["읍면동", "자격", "연령구간"]
)

full_combinations = (
    gangnam_welfare
    .set_index(["읍면동", "자격", "연령구간"])
    .reindex(full_index)
    .reset_index()
)

missing_combinations = full_combinations[
    full_combinations["수급권자수"].isna()
]

print("전체 가능한 조합:", len(full_combinations))
print("실제 기록된 조합:", len(gangnam_welfare))
print("누락 조합:", len(missing_combinations))

display(missing_combinations.head(30))

전체 가능한 조합: 264
실제 기록된 조합: 204
누락 조합: 60


,읍면동,자격,연령구간,수급권자수
3,신사동,기초의료급여,18세미만,NaN
11,신사동,기초교육급여,65세이상,NaN
15,논현1동,기초의료급여,18세미만,NaN
22,논현1동,기초교육급여,18~64세,NaN
23,논현1동,기초교육급여,65세이상,NaN
34,논현2동,기초교육급여,18~64세,NaN
35,논현2동,기초교육급여,65세이상,NaN
39,압구정동,기초의료급여,18세미만,NaN
46,압구정동,기초교육급여,18~64세,NaN
47,압구정동,기초교육급여,65세이상,NaN


In [13]:
duplicate_check = (
    gangnam_welfare
    .groupby(["읍면동", "자격", "연령구간"])
    .size()
    .reset_index(name="count")
)

duplicates = duplicate_check[
    duplicate_check["count"] > 1
]

print("중복 조합 수:", len(duplicates))

if len(duplicates) > 0:
    display(duplicates)
else:
    print("중복 없음")

중복 조합 수: 0
중복 없음


In [14]:
welfare_by_type = (
    gangnam_welfare
    .groupby(["읍면동", "자격"], as_index=False)
    .agg(수급권자수=("수급권자수", "sum"))
)

welfare_type_pivot = (
    welfare_by_type
    .pivot(
        index="읍면동",
        columns="자격",
        values="수급권자수"
    )
    .reset_index()
)

welfare_type_pivot.columns.name = None

print("크기:", welfare_type_pivot.shape)
print("결측치:")
print(welfare_type_pivot.isna().sum())

display(welfare_type_pivot)

크기: (22, 5)
결측치:
읍면동       0
기초교육급여    3
기초생계급여    0
기초의료급여    1
기초주거급여    0
dtype: int64


,읍면동,기초교육급여,기초생계급여,기초의료급여,기초주거급여
0,개포1동,2.0,99.0,5.0,44.0
1,개포2동,2.0,112.0,8.0,112.0
2,개포3동,6.0,1021.0,68.0,327.0
3,개포4동,11.0,519.0,39.0,223.0
4,논현1동,7.0,642.0,25.0,262.0
5,논현2동,6.0,326.0,25.0,201.0
6,대치1동,1.0,8.0,NaN,7.0
7,대치2동,1.0,71.0,6.0,91.0
8,대치4동,4.0,337.0,17.0,249.0
9,도곡1동,NaN,77.0,4.0,46.0


In [15]:
from google.colab import files

uploaded = files.upload()

Saving gangnam_resident_population_features_2025Q3_Q4.csv to gangnam_resident_population_features_2025Q3_Q4.csv


In [16]:
resident = pd.read_csv(
    "gangnam_resident_population_features_2025Q3_Q4.csv"
)

print("크기:", resident.shape)
print("\n컬럼:")
print(resident.columns.tolist())

print("\n분기:")
print(resident["quarter"].unique())

print("\n행정동 수:")
print(resident["dong_name"].nunique())

display(
    resident[
        [
            "dong_name",
            "quarter",
            "resident_total",
            "resident_age_65_plus"
        ]
    ].head(10)
)

크기: (44, 27)

컬럼:
['dong_name', 'quarter', 'resident_total', 'resident_korean', 'resident_foreigner', 'foreigner_ratio', 'resident_age_10', 'resident_age_20', 'resident_age_30', 'resident_age_40', 'resident_age_50', 'resident_age_60_plus', 'resident_age_65_plus', 'resident_age_10_ratio', 'resident_age_20_ratio', 'resident_age_30_ratio', 'resident_age_40_ratio', 'resident_age_50_ratio', 'resident_age_60_plus_ratio', 'resident_age_65_plus_ratio', 'resident_10plus', 'resident_age_10_ratio_10plus', 'resident_age_20_ratio_10plus', 'resident_age_30_ratio_10plus', 'resident_age_40_ratio_10plus', 'resident_age_50_ratio_10plus', 'resident_age_60_plus_ratio_10plus']

분기:
['2025Q3' '2025Q4']

행정동 수:
22


,dong_name,quarter,resident_total,resident_age_65_plus
0,신사동,2025Q3,15327,3238
1,신사동,2025Q4,15291,3233
2,논현1동,2025Q3,21571,3220
3,논현1동,2025Q4,21500,3232
4,논현2동,2025Q3,20511,3347
5,논현2동,2025Q4,20506,3395
6,압구정동,2025Q3,25671,5542
7,압구정동,2025Q4,25668,5566
8,청담동,2025Q3,24115,4212
9,청담동,2025Q4,25463,4617


In [17]:
resident_population_check = (
    resident
    .pivot(
        index="dong_name",
        columns="quarter",
        values="resident_total"
    )
    .reset_index()
)

resident_population_check["change"] = (
    resident_population_check["2025Q4"]
    - resident_population_check["2025Q3"]
)

resident_population_check["change_rate_pct"] = (
    resident_population_check["change"]
    / resident_population_check["2025Q3"] * 100
)

resident_population_check["resident_mean"] = (
    resident_population_check[["2025Q3", "2025Q4"]]
    .mean(axis=1)
)

display(
    resident_population_check.sort_values(
        "change_rate_pct",
        key=abs,
        ascending=False
    )
)

print(
    "\n평균 절대 변화율:",
    resident_population_check["change_rate_pct"].abs().mean()
)

print(
    "최대 절대 변화율:",
    resident_population_check["change_rate_pct"].abs().max()
)

quarter,dong_name,2025Q3,2025Q4,change,change_rate_pct,resident_mean
21,청담동,24115,25463,1348,5.589882,24789.0
2,개포3동,16592,15997,-595,-3.586066,16294.5
8,대치4동,18598,18326,-272,-1.462523,18462.0
19,일원1동,14298,14221,-77,-0.538537,14259.5
0,개포1동,26818,26681,-137,-0.510851,26749.5
11,삼성1동,13068,13005,-63,-0.482094,13036.5
4,논현1동,21571,21500,-71,-0.329146,21535.5
7,대치2동,38575,38694,119,0.308490,38634.5
15,신사동,15327,15291,-36,-0.234880,15309.0
10,도곡2동,32270,32199,-71,-0.220019,32234.5



평균 절대 변화율: 0.6591485661770796
최대 절대 변화율: 5.589881816296911


In [18]:
welfare_dongs = set(welfare_type_pivot["읍면동"])
resident_dongs = set(resident_population_check["dong_name"])

print("수급권자 행정동 수:", len(welfare_dongs))
print("등록인구 행정동 수:", len(resident_dongs))

print("\n수급권자에만 있는 동:")
print(welfare_dongs - resident_dongs)

print("\n등록인구에만 있는 동:")
print(resident_dongs - welfare_dongs)

수급권자 행정동 수: 22
등록인구 행정동 수: 22

수급권자에만 있는 동:
set()

등록인구에만 있는 동:
set()


In [19]:
economic_base = welfare_type_pivot.merge(
    resident_population_check[
        ["dong_name", "resident_mean"]
    ],
    left_on="읍면동",
    right_on="dong_name",
    how="left",
    validate="one_to_one"
)

economic_base = economic_base.drop(columns="dong_name")

print("결합 결과 크기:", economic_base.shape)
print("\n결측치:")
print(economic_base.isna().sum())

display(economic_base.head())

결합 결과 크기: (22, 6)

결측치:
읍면동              0
기초교육급여           3
기초생계급여           0
기초의료급여           1
기초주거급여           0
resident_mean    0
dtype: int64


,읍면동,기초교육급여,기초생계급여,기초의료급여,기초주거급여,resident_mean
0,개포1동,2.0,99.0,5.0,44.0,26749.5
1,개포2동,2.0,112.0,8.0,112.0,42223.5
2,개포3동,6.0,1021.0,68.0,327.0,16294.5
3,개포4동,11.0,519.0,39.0,223.0,23161.5
4,논현1동,7.0,642.0,25.0,262.0,21535.5


In [20]:
economic_feature = economic_base.copy()

economic_feature["livelihood_per_1000"] = (
    economic_feature["기초생계급여"]
    / economic_feature["resident_mean"] * 1000
)

economic_feature["medical_per_1000"] = (
    economic_feature["기초의료급여"]
    / economic_feature["resident_mean"] * 1000
)

economic_feature["housing_per_1000"] = (
    economic_feature["기초주거급여"]
    / economic_feature["resident_mean"] * 1000
)

economic_feature["education_per_1000"] = (
    economic_feature["기초교육급여"]
    / economic_feature["resident_mean"] * 1000
)

display(
    economic_feature[
        [
            "읍면동",
            "livelihood_per_1000",
            "medical_per_1000",
            "housing_per_1000",
            "education_per_1000"
        ]
    ].sort_values(
        "livelihood_per_1000",
        ascending=False
    )
)

,읍면동,livelihood_per_1000,medical_per_1000,housing_per_1000,education_per_1000
14,수서동,203.625400,11.618929,42.725301,0.220613
19,일원1동,83.803780,6.942740,30.576107,0.350643
2,개포3동,62.659179,4.173187,20.068121,0.368222
13,세곡동,36.821516,3.250870,20.935600,0.563484
4,논현1동,29.811242,1.160874,12.165959,0.325045
3,개포4동,22.407875,1.683829,9.628047,0.474926
17,역삼1동,21.872327,1.458155,9.646257,0.084124
8,대치4동,18.253710,0.920810,13.487163,0.216661
5,논현2동,15.895848,1.219007,9.800814,0.292562
12,삼성2동,11.079880,0.817100,5.556282,0.032684


In [21]:
rate_cols = [
    "livelihood_per_1000",
    "medical_per_1000",
    "housing_per_1000",
    "education_per_1000"
]

display(
    economic_feature[rate_cols]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
livelihood_per_1000,22.0,25.243327,45.044243,0.335810,3.665407,8.532220,22.273988,203.625400
medical_per_1000,21.0,1.741321,2.815023,0.062045,0.189468,0.817100,1.458155,11.618929
housing_per_1000,22.0,9.413403,10.702531,0.293834,2.429694,5.066690,11.574672,42.725301
education_per_1000,19.0,0.197667,0.162146,0.025884,0.054706,0.195963,0.308803,0.563484


In [22]:
elderly_welfare = (
    gangnam_welfare[
        gangnam_welfare["연령구간"] == "65세이상"
    ]
    .pivot(
        index="읍면동",
        columns="자격",
        values="수급권자수"
    )
    .reset_index()
)

elderly_welfare.columns.name = None

elderly_welfare = elderly_welfare.rename(columns={
    "기초생계급여": "livelihood_65plus",
    "기초의료급여": "medical_65plus",
    "기초주거급여": "housing_65plus",
    "기초교육급여": "education_65plus"
})

print("크기:", elderly_welfare.shape)

print("\n결측치:")
print(elderly_welfare.isna().sum())

display(elderly_welfare)

크기: (22, 4)

결측치:
읍면동                  0
livelihood_65plus    0
medical_65plus       1
housing_65plus       0
dtype: int64


,읍면동,livelihood_65plus,medical_65plus,housing_65plus
0,개포1동,65.0,5.0,32.0
1,개포2동,69.0,6.0,43.0
2,개포3동,641.0,37.0,262.0
3,개포4동,197.0,18.0,73.0
4,논현1동,217.0,9.0,61.0
5,논현2동,104.0,8.0,39.0
6,대치1동,3.0,NaN,2.0
7,대치2동,31.0,2.0,25.0
8,대치4동,111.0,4.0,50.0
9,도곡1동,29.0,1.0,17.0


In [23]:
working_age_welfare = (
    gangnam_welfare[
        gangnam_welfare["연령구간"] == "18~64세"
    ]
    .pivot(
        index="읍면동",
        columns="자격",
        values="수급권자수"
    )
    .reset_index()
)

working_age_welfare.columns.name = None

working_age_welfare = working_age_welfare.rename(columns={
    "기초생계급여": "livelihood_18_64",
    "기초의료급여": "medical_18_64",
    "기초주거급여": "housing_18_64",
    "기초교육급여": "education_18_64"
})

print("크기:", working_age_welfare.shape)

print("\n결측치:")
print(working_age_welfare.isna().sum())

display(working_age_welfare.head(22))

크기: (22, 5)

결측치:
읍면동                  0
education_18_64     14
livelihood_18_64     0
medical_18_64        4
housing_18_64        0
dtype: int64


,읍면동,education_18_64,livelihood_18_64,medical_18_64,housing_18_64
0,개포1동,2.0,29.0,NaN,10.0
1,개포2동,NaN,40.0,2.0,56.0
2,개포3동,NaN,366.0,30.0,56.0
3,개포4동,2.0,258.0,21.0,130.0
4,논현1동,NaN,385.0,16.0,175.0
5,논현2동,NaN,199.0,15.0,146.0
6,대치1동,NaN,5.0,NaN,5.0
7,대치2동,NaN,30.0,4.0,55.0
8,대치4동,1.0,194.0,11.0,170.0
9,도곡1동,NaN,48.0,3.0,29.0


In [24]:
resident_65plus = (
    resident
    .pivot(
        index="dong_name",
        columns="quarter",
        values="resident_age_65_plus"
    )
    .reset_index()
)

resident_65plus["resident_65plus_mean"] = (
    resident_65plus[["2025Q3", "2025Q4"]]
    .mean(axis=1)
)

display(resident_65plus.head())

print(
    "Q3→Q4 평균 절대 변화율:",
    (
        (
            resident_65plus["2025Q4"]
            - resident_65plus["2025Q3"]
        )
        / resident_65plus["2025Q3"] * 100
    ).abs().mean()
)

quarter,dong_name,2025Q3,2025Q4,resident_65plus_mean
0,개포1동,4407,4467,4437.0
1,개포2동,6016,6061,6038.5
2,개포3동,3801,3726,3763.5
3,개포4동,4003,4065,4034.0
4,논현1동,3220,3232,3226.0


Q3→Q4 평균 절대 변화율: 1.3772637598048467


In [25]:
elderly_economic = elderly_welfare.merge(
    resident_65plus[
        ["dong_name", "resident_65plus_mean"]
    ],
    left_on="읍면동",
    right_on="dong_name",
    how="left",
    validate="one_to_one"
).drop(columns="dong_name")

print("크기:", elderly_economic.shape)
print("\n결측치:")
print(elderly_economic.isna().sum())

display(elderly_economic.head())

크기: (22, 5)

결측치:
읍면동                     0
livelihood_65plus       0
medical_65plus          1
housing_65plus          0
resident_65plus_mean    0
dtype: int64


,읍면동,livelihood_65plus,medical_65plus,housing_65plus,resident_65plus_mean
0,개포1동,65.0,5.0,32.0,4437.0
1,개포2동,69.0,6.0,43.0,6038.5
2,개포3동,641.0,37.0,262.0,3763.5
3,개포4동,197.0,18.0,73.0,4034.0
4,논현1동,217.0,9.0,61.0,3226.0


In [26]:
elderly_economic["elderly_livelihood_per_1000"] = (
    elderly_economic["livelihood_65plus"]
    / elderly_economic["resident_65plus_mean"] * 1000
)

elderly_economic["elderly_medical_per_1000"] = (
    elderly_economic["medical_65plus"]
    / elderly_economic["resident_65plus_mean"] * 1000
)

elderly_economic["elderly_housing_per_1000"] = (
    elderly_economic["housing_65plus"]
    / elderly_economic["resident_65plus_mean"] * 1000
)

display(
    elderly_economic[
        [
            "읍면동",
            "elderly_livelihood_per_1000",
            "elderly_medical_per_1000",
            "elderly_housing_per_1000"
        ]
    ].sort_values(
        "elderly_livelihood_per_1000",
        ascending=False
    )
)

,읍면동,elderly_livelihood_per_1000,elderly_medical_per_1000,elderly_housing_per_1000
14,수서동,325.228807,15.346214,80.613849
2,개포3동,170.320181,9.831274,69.616049
19,일원1동,168.935889,12.690020,55.254461
13,세곡동,74.051408,5.140759,48.102815
4,논현1동,67.265964,2.789833,18.908865
17,역삼1동,59.257468,3.628008,17.414439
3,개포4동,48.834903,4.462072,18.096182
8,대치4동,48.177083,1.736111,21.701389
12,삼성2동,34.490849,1.407790,8.916002
5,논현2동,30.851379,2.373183,11.569267


In [27]:
elderly_rate_cols = [
    "elderly_livelihood_per_1000",
    "elderly_medical_per_1000",
    "elderly_housing_per_1000"
]

display(
    elderly_economic[elderly_rate_cols]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
elderly_livelihood_per_1000,22.0,52.188413,77.292451,1.020582,9.175670,20.546639,56.651827,325.228807
elderly_medical_per_1000,21.0,3.297903,4.235778,0.180050,0.453052,1.736111,3.628008,15.346214
elderly_housing_per_1000,22.0,18.748071,22.934980,0.680388,4.883915,8.630463,18.705695,80.613849


In [28]:
welfare_feature = (
    economic_feature[
        [
            "읍면동",
            "livelihood_per_1000",
            "medical_per_1000",
            "housing_per_1000"
        ]
    ]
    .merge(
        elderly_economic[
            [
                "읍면동",
                "elderly_livelihood_per_1000",
                "elderly_medical_per_1000",
                "elderly_housing_per_1000"
            ]
        ],
        on="읍면동",
        how="left",
        validate="one_to_one"
    )
)

welfare_feature = welfare_feature.rename(
    columns={"읍면동": "dong_name"}
)

print("크기:", welfare_feature.shape)

print("\n결측치:")
print(welfare_feature.isna().sum())

display(
    welfare_feature.sort_values(
        "elderly_livelihood_per_1000",
        ascending=False
    )
)

크기: (22, 7)

결측치:
dong_name                      0
livelihood_per_1000            0
medical_per_1000               1
housing_per_1000               0
elderly_livelihood_per_1000    0
elderly_medical_per_1000       1
elderly_housing_per_1000       0
dtype: int64


,dong_name,livelihood_per_1000,medical_per_1000,housing_per_1000,elderly_livelihood_per_1000,elderly_medical_per_1000,elderly_housing_per_1000
14,수서동,203.625400,11.618929,42.725301,325.228807,15.346214,80.613849
2,개포3동,62.659179,4.173187,20.068121,170.320181,9.831274,69.616049
19,일원1동,83.803780,6.942740,30.576107,168.935889,12.690020,55.254461
13,세곡동,36.821516,3.250870,20.935600,74.051408,5.140759,48.102815
4,논현1동,29.811242,1.160874,12.165959,67.265964,2.789833,18.908865
17,역삼1동,21.872327,1.458155,9.646257,59.257468,3.628008,17.414439
3,개포4동,22.407875,1.683829,9.628047,48.834903,4.462072,18.096182
8,대치4동,18.253710,0.920810,13.487163,48.177083,1.736111,21.701389
12,삼성2동,11.079880,0.817100,5.556282,34.490849,1.407790,8.916002
5,논현2동,15.895848,1.219007,9.800814,30.851379,2.373183,11.569267


In [29]:
# 소수점 정리
numeric_cols = welfare_feature.columns.drop("dong_name")

welfare_feature[numeric_cols] = (
    welfare_feature[numeric_cols]
    .round(4)
)

# 행정동명 기준 정렬
welfare_feature = (
    welfare_feature
    .sort_values("dong_name")
    .reset_index(drop=True)
)

output_file = "gangnam_welfare_economic_features_2024_05.csv"

welfare_feature.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_file)
print("크기:", welfare_feature.shape)
print("\n결측치:")
print(welfare_feature.isna().sum())

display(welfare_feature.head())

저장 완료: gangnam_welfare_economic_features_2024_05.csv
크기: (22, 7)

결측치:
dong_name                      0
livelihood_per_1000            0
medical_per_1000               1
housing_per_1000               0
elderly_livelihood_per_1000    0
elderly_medical_per_1000       1
elderly_housing_per_1000       0
dtype: int64


,dong_name,livelihood_per_1000,medical_per_1000,housing_per_1000,elderly_livelihood_per_1000,elderly_medical_per_1000,elderly_housing_per_1000
0,개포1동,3.7010,0.1869,1.6449,14.6495,1.1269,7.2121
1,개포2동,2.6526,0.1895,2.6526,11.4267,0.9936,7.1210
2,개포3동,62.6592,4.1732,20.0681,170.3202,9.8313,69.6160
3,개포4동,22.4079,1.6838,9.6280,48.8349,4.4621,18.0962
4,논현1동,29.8112,1.1609,12.1660,67.2660,2.7898,18.9089


In [30]:
from google.colab import files

files.download(
    "gangnam_welfare_economic_features_2024_05.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
from google.colab import files

uploaded = files.upload()

Saving 국민기초생활보장+수급자(2020+이후)_20260922173232.csv to 국민기초생활보장+수급자(2020+이후)_20260922173232.csv


In [32]:
print(list(uploaded.keys()))

['국민기초생활보장+수급자(2020+이후)_20260922173232.csv']


In [34]:
import os

print("업로드된 파일:")
for name in uploaded.keys():
    print(
        "파일명:", repr(name),
        "| 확장자:", os.path.splitext(name)[1],
        "| 크기:", os.path.getsize(name), "bytes"
    )

업로드된 파일:
파일명: '국민기초생활보장+수급자(2020+이후)_20260922173232.csv' | 확장자: .csv | 크기: 3121 bytes


In [35]:
print("\n현재 Colab 파일 목록:")
print(os.listdir("/content"))


현재 Colab 파일 목록:
['.config', '국민기초생활보장+수급자(2020+이후)_20260922173232.csv', 'gangnam_welfare_economic_features_2024_05.csv', '국민기초생활 수급자 동별 현황(202405).xlsx', 'gangnam_resident_population_features_2025Q3_Q4.csv', 'sample_data']


In [36]:
import pandas as pd

filename = "국민기초생활보장+수급자(2020+이후)_20260922173232.csv"

welfare2_raw = pd.read_csv(
    filename,
    encoding="utf-8-sig"
)

print("원본 크기:", welfare2_raw.shape)

print("\n컬럼:")
print(welfare2_raw.columns.tolist())

display(welfare2_raw.head(20))

원본 크기: (27, 24)

컬럼:
['동별(1)', '동별(2)', '2024', '2024.1', '2024.2', '2024.3', '2024.4', '2024.5', '2024.6', '2024.7', '2024.8', '2024.9', '2024.10', '2024.11', '2024.12', '2024.13', '2024.14', '2024.15', '2024.16', '2024.17', '2024.18', '2024.19', '2024.20', '2024.21']


,동별(1),동별(2),2024,2024.1,2024.2,2024.3,2024.4,2024.5,2024.6,2024.7,...,2024.12,2024.13,2024.14,2024.15,2024.16,2024.17,2024.18,2024.19,2024.20,2024.21
0,동별(1),동별(2),총 수급자,총 수급자,총 수급자,총 수급자,총 수급자,일반수급자,일반수급자,일반수급자,...,조건부 수급자,특례수급자,특례수급자,특례수급자,특례수급자,시설수급자,시설수급자,시설수급자,시설수급자,시설수급자
1,동별(1),동별(2),가구 (가구),시설수 (개소),인원 (명),인원 (명),인원 (명),가구 (가구),인원 (명),인원 (명),...,인원 (명),가구 (가구),인원 (명),인원 (명),인원 (명),시설수 (개소),가구 (가구),인원 (명),인원 (명),인원 (명)
2,동별(1),동별(2),소계,소계,소계,남자,여자,소계,소계,남자,...,여자,소계,소계,남자,여자,소계,소계,소계,남자,여자
3,강남구,소계,12114,13,16959,7353,9606,9955,12700,5402,...,1718,179,262,112,150,13,563,563,123,440
4,강남구,신사동,170,-,221,98,123,143,181,79,...,19,1,3,1,2,-,-,-,-,-
5,강남구,논현1동,736,-,916,424,492,535,640,278,...,125,9,11,6,5,-,-,-,-,-
6,강남구,논현2동,423,-,571,259,312,318,409,168,...,69,3,4,2,2,-,-,-,-,-
7,강남구,압구정동,109,-,120,53,67,90,97,40,...,10,1,1,1,-,-,-,-,-,-
8,강남구,청담동,172,-,241,111,130,137,180,77,...,27,-,-,-,-,-,-,-,-,-
9,강남구,삼성1동,90,-,136,65,71,66,96,44,...,19,1,1,1,-,-,-,-,-,-


In [38]:
header_info = pd.DataFrame({
    "column": welfare2_raw.columns,
    "level_1": welfare2_raw.iloc[0].values,
    "level_2": welfare2_raw.iloc[1].values,
    "level_3": welfare2_raw.iloc[2].values
})

display(header_info)

,column,level_1,level_2,level_3
0,동별(1),동별(1),동별(1),동별(1)
1,동별(2),동별(2),동별(2),동별(2)
2,2024,총 수급자,가구 (가구),소계
3,2024.1,총 수급자,시설수 (개소),소계
4,2024.2,총 수급자,인원 (명),소계
5,2024.3,총 수급자,인원 (명),남자
6,2024.4,총 수급자,인원 (명),여자
7,2024.5,일반수급자,가구 (가구),소계
8,2024.6,일반수급자,인원 (명),소계
9,2024.7,일반수급자,인원 (명),남자


In [39]:
print("동별(1) 고유값:")
print(welfare2_raw.iloc[3:]["동별(1)"].unique())

print("\n강남구 행정동:")
print(
    welfare2_raw.iloc[3:][
        welfare2_raw.iloc[3:]["동별(1)"] == "강남구"
    ]["동별(2)"].tolist()
)

동별(1) 고유값:
['강남구']

강남구 행정동:
['소계', '신사동', '논현1동', '논현2동', '압구정동', '청담동', '삼성1동', '삼성2동', '대치1동', '대치2동', '대치4동', '역삼1동', '역삼2동', '도곡1동', '도곡2동', '개포1동', '개포2동', '개포3동', '개포4동', '세곡동', '일원본동', '일원1동', '수서동', '기타']


In [40]:
welfare2 = welfare2_raw.iloc[3:].copy()

# 소계와 기타 제외 → 강남구 22개 행정동
welfare2 = welfare2[
    ~welfare2["동별(2)"].isin(["소계", "기타"])
].copy()

welfare2 = welfare2[
    [
        "동별(2)",
        "2024.2",   # 총 수급자 인원
        "2024.3",   # 총 수급자 남자
        "2024.4",   # 총 수급자 여자
        "2024.6",   # 일반수급자 인원
        "2024.10",  # 조건부 수급자 인원
        "2024.14",  # 특례수급자 인원
        "2024.19"   # 시설수급자 인원
    ]
].copy()

welfare2.columns = [
    "dong_name",
    "total_recipient",
    "total_recipient_male",
    "total_recipient_female",
    "general_recipient",
    "conditional_recipient",
    "special_recipient",
    "facility_recipient"
]

numeric_cols = welfare2.columns.drop("dong_name")

for col in numeric_cols:
    welfare2[col] = pd.to_numeric(
        welfare2[col].replace("-", 0),
        errors="coerce"
    )

print("크기:", welfare2.shape)
print("\n행정동 수:", welfare2["dong_name"].nunique())

print("\n결측치:")
print(welfare2.isna().sum())

display(welfare2)

크기: (22, 8)

행정동 수: 22

결측치:
dong_name                 0
total_recipient           0
total_recipient_male      0
total_recipient_female    0
general_recipient         0
conditional_recipient     0
special_recipient         0
facility_recipient        0
dtype: int64


/tmp/ipykernel_2140/1467017628.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  welfare2[col].replace("-", 0),


,dong_name,total_recipient,total_recipient_male,total_recipient_female,general_recipient,conditional_recipient,special_recipient,facility_recipient
4,신사동,221,98,123,181,37,3,0
5,논현1동,916,424,492,640,265,11,0
6,논현2동,571,259,312,409,158,4,0
7,압구정동,120,53,67,97,22,1,0
8,청담동,241,111,130,180,61,0,0
9,삼성1동,136,65,71,96,39,1,0
10,삼성2동,541,255,286,407,133,1,0
11,대치1동,17,9,8,13,4,0,0
12,대치2동,173,81,92,132,39,2,0
13,대치4동,618,292,326,440,173,5,0


In [41]:
welfare2["gender_sum"] = (
    welfare2["total_recipient_male"]
    + welfare2["total_recipient_female"]
)

welfare2["gender_diff"] = (
    welfare2["total_recipient"]
    - welfare2["gender_sum"]
)

print("성별 합계 불일치 행 수:")
print((welfare2["gender_diff"] != 0).sum())

display(
    welfare2[
        [
            "dong_name",
            "total_recipient",
            "total_recipient_male",
            "total_recipient_female",
            "gender_sum",
            "gender_diff"
        ]
    ]
)

성별 합계 불일치 행 수:
0


,dong_name,total_recipient,total_recipient_male,total_recipient_female,gender_sum,gender_diff
4,신사동,221,98,123,221,0
5,논현1동,916,424,492,916,0
6,논현2동,571,259,312,571,0
7,압구정동,120,53,67,120,0
8,청담동,241,111,130,241,0
9,삼성1동,136,65,71,136,0
10,삼성2동,541,255,286,541,0
11,대치1동,17,9,8,17,0
12,대치2동,173,81,92,173,0
13,대치4동,618,292,326,618,0


In [42]:
welfare2["type_sum"] = (
    welfare2["general_recipient"]
    + welfare2["conditional_recipient"]
    + welfare2["special_recipient"]
    + welfare2["facility_recipient"]
)

welfare2["type_diff"] = (
    welfare2["total_recipient"]
    - welfare2["type_sum"]
)

print("유형 합계 불일치 행 수:")
print((welfare2["type_diff"] != 0).sum())

display(
    welfare2[
        [
            "dong_name",
            "total_recipient",
            "general_recipient",
            "conditional_recipient",
            "special_recipient",
            "facility_recipient",
            "type_sum",
            "type_diff"
        ]
    ]
)

유형 합계 불일치 행 수:
0


,dong_name,total_recipient,general_recipient,conditional_recipient,special_recipient,facility_recipient,type_sum,type_diff
4,신사동,221,181,37,3,0,221,0
5,논현1동,916,640,265,11,0,916,0
6,논현2동,571,409,158,4,0,571,0
7,압구정동,120,97,22,1,0,120,0
8,청담동,241,180,61,0,0,241,0
9,삼성1동,136,96,39,1,0,136,0
10,삼성2동,541,407,133,1,0,541,0
11,대치1동,17,13,4,0,0,17,0
12,대치2동,173,132,39,2,0,173,0
13,대치4동,618,440,173,5,0,618,0


In [43]:
print("시설수급자 고유값:")
print(welfare2["facility_recipient"].unique())

print(
    "\n시설수급자가 0보다 큰 행정동 수:",
    (welfare2["facility_recipient"] > 0).sum()
)

시설수급자 고유값:
[0]

시설수급자가 0보다 큰 행정동 수: 0


In [44]:
welfare2_feature = welfare2.merge(
    resident_population_check[
        ["dong_name", "resident_mean"]
    ],
    on="dong_name",
    how="left",
    validate="one_to_one"
)

print("결합 크기:", welfare2_feature.shape)
print("resident_mean 결측치:", welfare2_feature["resident_mean"].isna().sum())

결합 크기: (22, 13)
resident_mean 결측치: 0


In [45]:
# 전체 인구 1,000명당 총 수급자
welfare2_feature["total_recipient_per_1000"] = (
    welfare2_feature["total_recipient"]
    / welfare2_feature["resident_mean"] * 1000
)

# 수급자 성별 구성비
welfare2_feature["recipient_male_ratio"] = (
    welfare2_feature["total_recipient_male"]
    / welfare2_feature["total_recipient"]
)

welfare2_feature["recipient_female_ratio"] = (
    welfare2_feature["total_recipient_female"]
    / welfare2_feature["total_recipient"]
)

# 수급자 유형 구성비
welfare2_feature["general_recipient_ratio"] = (
    welfare2_feature["general_recipient"]
    / welfare2_feature["total_recipient"]
)

welfare2_feature["conditional_recipient_ratio"] = (
    welfare2_feature["conditional_recipient"]
    / welfare2_feature["total_recipient"]
)

welfare2_feature["special_recipient_ratio"] = (
    welfare2_feature["special_recipient"]
    / welfare2_feature["total_recipient"]
)

In [46]:
welfare2_feature["gender_ratio_sum"] = (
    welfare2_feature["recipient_male_ratio"]
    + welfare2_feature["recipient_female_ratio"]
)

welfare2_feature["type_ratio_sum"] = (
    welfare2_feature["general_recipient_ratio"]
    + welfare2_feature["conditional_recipient_ratio"]
    + welfare2_feature["special_recipient_ratio"]
)

print(
    "성별 구성비 합 최대 오차:",
    (welfare2_feature["gender_ratio_sum"] - 1).abs().max()
)

print(
    "유형 구성비 합 최대 오차:",
    (welfare2_feature["type_ratio_sum"] - 1).abs().max()
)

display(
    welfare2_feature[
        [
            "dong_name",
            "total_recipient_per_1000",
            "recipient_male_ratio",
            "recipient_female_ratio",
            "general_recipient_ratio",
            "conditional_recipient_ratio",
            "special_recipient_ratio"
        ]
    ].sort_values(
        "total_recipient_per_1000",
        ascending=False
    )
)

성별 구성비 합 최대 오차: 0.0
유형 구성비 합 최대 오차: 1.1102230246251565e-16


,dong_name,total_recipient_per_1000,recipient_male_ratio,recipient_female_ratio,general_recipient_ratio,conditional_recipient_ratio,special_recipient_ratio
21,수서동,255.910578,0.422126,0.577874,0.877874,0.099138,0.022989
20,일원1동,124.127774,0.437853,0.562147,0.789831,0.180226,0.029944
16,개포3동,86.164043,0.363248,0.636752,0.879630,0.104701,0.015670
18,세곡동,62.460041,0.414990,0.585010,0.718251,0.267176,0.014573
1,논현1동,42.534420,0.462882,0.537118,0.698690,0.289301,0.012009
10,역삼1동,34.995724,0.533654,0.466346,0.637019,0.355769,0.007212
9,대치4동,33.474163,0.472492,0.527508,0.711974,0.279935,0.008091
17,개포4동,33.201649,0.440832,0.559168,0.723017,0.249675,0.027308
2,논현2동,27.842114,0.453590,0.546410,0.716287,0.276708,0.007005
6,삼성2동,17.682050,0.471349,0.528651,0.752311,0.245841,0.001848


In [47]:
# 첫 번째 경제취약 Feature 파일 불러오기
welfare1_feature = pd.read_csv(
    "gangnam_welfare_economic_features_2024_05.csv"
)

print("첫 번째 Feature 컬럼:")
print(welfare1_feature.columns.tolist())

display(welfare1_feature.head())

첫 번째 Feature 컬럼:
['dong_name', 'livelihood_per_1000', 'medical_per_1000', 'housing_per_1000', 'elderly_livelihood_per_1000', 'elderly_medical_per_1000', 'elderly_housing_per_1000']


,dong_name,livelihood_per_1000,medical_per_1000,housing_per_1000,elderly_livelihood_per_1000,elderly_medical_per_1000,elderly_housing_per_1000
0,개포1동,3.7010,0.1869,1.6449,14.6495,1.1269,7.2121
1,개포2동,2.6526,0.1895,2.6526,11.4267,0.9936,7.1210
2,개포3동,62.6592,4.1732,20.0681,170.3202,9.8313,69.6160
3,개포4동,22.4079,1.6838,9.6280,48.8349,4.4621,18.0962
4,논현1동,29.8112,1.1609,12.1660,67.2660,2.7898,18.9089


In [48]:
welfare_compare = welfare1_feature.merge(
    welfare2_feature[
        [
            "dong_name",
            "total_recipient_per_1000",
            "recipient_male_ratio",
            "recipient_female_ratio",
            "general_recipient_ratio",
            "conditional_recipient_ratio",
            "special_recipient_ratio"
        ]
    ],
    on="dong_name",
    how="inner",
    validate="one_to_one"
)

print("결합 크기:", welfare_compare.shape)
print("행정동 수:", welfare_compare["dong_name"].nunique())
print("\n결측치:")
print(welfare_compare.isna().sum())

display(welfare_compare.head())

결합 크기: (22, 13)
행정동 수: 22

결측치:
dong_name                      0
livelihood_per_1000            0
medical_per_1000               1
housing_per_1000               0
elderly_livelihood_per_1000    0
elderly_medical_per_1000       1
elderly_housing_per_1000       0
total_recipient_per_1000       0
recipient_male_ratio           0
recipient_female_ratio         0
general_recipient_ratio        0
conditional_recipient_ratio    0
special_recipient_ratio        0
dtype: int64


,dong_name,livelihood_per_1000,medical_per_1000,housing_per_1000,elderly_livelihood_per_1000,elderly_medical_per_1000,elderly_housing_per_1000,total_recipient_per_1000,recipient_male_ratio,recipient_female_ratio,general_recipient_ratio,conditional_recipient_ratio,special_recipient_ratio
0,개포1동,3.7010,0.1869,1.6449,14.6495,1.1269,7.2121,5.383278,0.430556,0.569444,0.840278,0.145833,0.013889
1,개포2동,2.6526,0.1895,2.6526,11.4267,0.9936,7.1210,6.134025,0.447876,0.552124,0.822394,0.169884,0.007722
2,개포3동,62.6592,4.1732,20.0681,170.3202,9.8313,69.6160,86.164043,0.363248,0.636752,0.879630,0.104701,0.015670
3,개포4동,22.4079,1.6838,9.6280,48.8349,4.4621,18.0962,33.201649,0.440832,0.559168,0.723017,0.249675,0.027308
4,논현1동,29.8112,1.1609,12.1660,67.2660,2.7898,18.9089,42.534420,0.462882,0.537118,0.698690,0.289301,0.012009


In [49]:
compare_cols = [
    "total_recipient_per_1000",
    "livelihood_per_1000",
    "medical_per_1000",
    "housing_per_1000",
    "elderly_livelihood_per_1000",
    "elderly_medical_per_1000",
    "elderly_housing_per_1000"
]

corr = welfare_compare[compare_cols].corr()

print("total_recipient_per_1000과의 상관계수:")
display(
    corr[["total_recipient_per_1000"]]
    .sort_values("total_recipient_per_1000", ascending=False)
)

total_recipient_per_1000과의 상관계수:


,total_recipient_per_1000
total_recipient_per_1000,1.000000
livelihood_per_1000,0.996414
medical_per_1000,0.991300
elderly_livelihood_per_1000,0.985042
housing_per_1000,0.962678
elderly_medical_per_1000,0.940150
elderly_housing_per_1000,0.905642


In [50]:
welfare2_final = welfare2_feature[
    [
        "dong_name",
        "total_recipient_per_1000",
        "recipient_male_ratio",
        "recipient_female_ratio",
        "general_recipient_ratio",
        "conditional_recipient_ratio",
        "special_recipient_ratio"
    ]
].copy()

# 소수점 정리
numeric_cols = welfare2_final.columns.drop("dong_name")
welfare2_final[numeric_cols] = welfare2_final[numeric_cols].round(4)

print("최종 크기:", welfare2_final.shape)
print("\n결측치:")
print(welfare2_final.isna().sum())

display(welfare2_final)

최종 크기: (22, 7)

결측치:
dong_name                      0
total_recipient_per_1000       0
recipient_male_ratio           0
recipient_female_ratio         0
general_recipient_ratio        0
conditional_recipient_ratio    0
special_recipient_ratio        0
dtype: int64


,dong_name,total_recipient_per_1000,recipient_male_ratio,recipient_female_ratio,general_recipient_ratio,conditional_recipient_ratio,special_recipient_ratio
0,신사동,14.4360,0.4434,0.5566,0.8190,0.1674,0.0136
1,논현1동,42.5344,0.4629,0.5371,0.6987,0.2893,0.0120
2,논현2동,27.8421,0.4536,0.5464,0.7163,0.2767,0.0070
3,압구정동,4.6748,0.4417,0.5583,0.8083,0.1833,0.0083
4,청담동,9.7221,0.4606,0.5394,0.7469,0.2531,0.0000
5,삼성1동,10.4322,0.4779,0.5221,0.7059,0.2868,0.0074
6,삼성2동,17.6820,0.4713,0.5287,0.7523,0.2458,0.0018
7,대치1동,0.7136,0.5294,0.4706,0.7647,0.2353,0.0000
8,대치2동,4.4779,0.4682,0.5318,0.7630,0.2254,0.0116
9,대치4동,33.4742,0.4725,0.5275,0.7120,0.2799,0.0081


In [51]:
output_file = "gangnam_welfare_recipient_features_2024.csv"

welfare2_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_file)
print("shape:", welfare2_final.shape)

저장 완료: gangnam_welfare_recipient_features_2024.csv
shape: (22, 7)


In [52]:
from google.colab import files

files.download("gangnam_welfare_recipient_features_2024.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [53]:
from google.colab import files

uploaded = files.upload()

Saving 국민기초생활보장+연령별+일반수급자(구별)_20260922173516.csv to 국민기초생활보장+연령별+일반수급자(구별)_20260922173516.csv


In [54]:
print(list(uploaded.keys()))

['국민기초생활보장+연령별+일반수급자(구별)_20260922173516.csv']


In [55]:
import os

filename = list(uploaded.keys())[0]

print("파일명:", filename)
print("확장자:", os.path.splitext(filename)[1])
print("파일 크기:", os.path.getsize(filename), "bytes")

파일명: 국민기초생활보장+연령별+일반수급자(구별)_20260922173516.csv
확장자: .csv
파일 크기: 494 bytes


In [58]:
age_welfare_raw = pd.read_csv(
    filename,
    encoding="cp949"
)

print("원본 크기:", age_welfare_raw.shape)

print("\n컬럼:")
print(age_welfare_raw.columns.tolist())

display(age_welfare_raw)

원본 크기: (2, 1)

컬럼:
['시점\t자치구별(1)\t자치구별(2)\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자']


,시점\t자치구별(1)\t자치구별(2)\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자\t일반수급자
0,시점\t자치구별(1)\t자치구별(2)\t소계\t0~4세\t5~9세\t10~14세\t...
1,2024\t합계\t강남구\t16396\t78\t184\t429\t651\t594\t...


In [59]:
age_welfare_raw = pd.read_csv(
    filename,
    encoding="cp949",
    sep="\t"
)

print("원본 크기:", age_welfare_raw.shape)

print("\n컬럼:")
print(age_welfare_raw.columns.tolist())

display(age_welfare_raw)

원본 크기: (2, 21)

컬럼:
['시점', '자치구별(1)', '자치구별(2)', '일반수급자', '일반수급자.1', '일반수급자.2', '일반수급자.3', '일반수급자.4', '일반수급자.5', '일반수급자.6', '일반수급자.7', '일반수급자.8', '일반수급자.9', '일반수급자.10', '일반수급자.11', '일반수급자.12', '일반수급자.13', '일반수급자.14', '일반수급자.15', '일반수급자.16', '일반수급자.17']


,시점,자치구별(1),자치구별(2),일반수급자,일반수급자.1,일반수급자.2,일반수급자.3,일반수급자.4,일반수급자.5,일반수급자.6,...,일반수급자.8,일반수급자.9,일반수급자.10,일반수급자.11,일반수급자.12,일반수급자.13,일반수급자.14,일반수급자.15,일반수급자.16,일반수급자.17
0,시점,자치구별(1),자치구별(2),소계,0~4세,5~9세,10~14세,15~19세,20~24세,25~29세,...,35~39세,40~44세,45~49세,50~54세,55~59세,60~64세,65~69세,70~74세,75~79세,80세이상
1,2024,합계,강남구,16396,78,184,429,651,594,380,...,443,740,817,1126,1309,1658,2170,1715,1595,2082


In [60]:
# 실제 연령구간 이름
age_labels = age_welfare_raw.iloc[0, 3:].tolist()

# 실제 2024년 강남구 데이터
age_values = age_welfare_raw.iloc[1, 3:].tolist()

gangnam_age_welfare = pd.DataFrame({
    "age_group": age_labels,
    "general_recipient": age_values
})

gangnam_age_welfare["general_recipient"] = pd.to_numeric(
    gangnam_age_welfare["general_recipient"],
    errors="coerce"
)

print("크기:", gangnam_age_welfare.shape)
print("\n결측치:")
print(gangnam_age_welfare.isna().sum())

display(gangnam_age_welfare)

크기: (18, 2)

결측치:
age_group            0
general_recipient    0
dtype: int64


,age_group,general_recipient
0,소계,16396
1,0~4세,78
2,5~9세,184
3,10~14세,429
4,15~19세,651
5,20~24세,594
6,25~29세,380
7,30~34세,425
8,35~39세,443
9,40~44세,740


In [61]:
total_reported = gangnam_age_welfare.loc[
    gangnam_age_welfare["age_group"] == "소계",
    "general_recipient"
].iloc[0]

age_sum = gangnam_age_welfare.loc[
    gangnam_age_welfare["age_group"] != "소계",
    "general_recipient"
].sum()

print("원자료 소계:", total_reported)
print("연령별 합계:", age_sum)
print("차이:", total_reported - age_sum)

원자료 소계: 16396
연령별 합계: 16396
차이: 0


In [62]:
# 소계 제외
age_detail = gangnam_age_welfare[
    gangnam_age_welfare["age_group"] != "소계"
].copy()

# 연령구간별 합산
age_0_19 = age_detail[
    age_detail["age_group"].isin(
        ["0~4세", "5~9세", "10~14세", "15~19세"]
    )
]["general_recipient"].sum()

age_20_64 = age_detail[
    age_detail["age_group"].isin(
        [
            "20~24세", "25~29세", "30~34세", "35~39세",
            "40~44세", "45~49세", "50~54세",
            "55~59세", "60~64세"
        ]
    )
]["general_recipient"].sum()

age_65_plus = age_detail[
    age_detail["age_group"].isin(
        ["65~69세", "70~74세", "75~79세", "80세이상"]
    )
]["general_recipient"].sum()

age_80_plus = age_detail.loc[
    age_detail["age_group"] == "80세이상",
    "general_recipient"
].iloc[0]

total = total_reported

gangnam_welfare_age_summary = pd.DataFrame({
    "year": [2024],
    "region": ["강남구"],
    "general_recipient_total": [total],
    "recipient_0_19": [age_0_19],
    "recipient_0_19_ratio": [age_0_19 / total],
    "recipient_20_64": [age_20_64],
    "recipient_20_64_ratio": [age_20_64 / total],
    "recipient_65_plus": [age_65_plus],
    "recipient_65_plus_ratio": [age_65_plus / total],
    "recipient_80_plus": [age_80_plus],
    "recipient_80_plus_ratio": [age_80_plus / total]
})

display(gangnam_welfare_age_summary.round(4))

,year,region,general_recipient_total,recipient_0_19,recipient_0_19_ratio,recipient_20_64,recipient_20_64_ratio,recipient_65_plus,recipient_65_plus_ratio,recipient_80_plus,recipient_80_plus_ratio
0,2024,강남구,16396,1342,0.0818,7492,0.4569,7562,0.4612,2082,0.127


In [63]:
print(
    "3개 연령집단 합:",
    age_0_19 + age_20_64 + age_65_plus
)

print(
    "원자료 총 수급자:",
    total
)

print(
    "차이:",
    total - (age_0_19 + age_20_64 + age_65_plus)
)

3개 연령집단 합: 16396
원자료 총 수급자: 16396
차이: 0


In [64]:
# 5세 단위 상세자료 저장
age_detail_output = "gangnam_general_recipient_age_detail_2024.csv"

age_detail.to_csv(
    age_detail_output,
    index=False,
    encoding="utf-8-sig"
)

# 연령대 요약자료 저장
age_summary_output = "gangnam_general_recipient_age_summary_2024.csv"

gangnam_welfare_age_summary.to_csv(
    age_summary_output,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:")
print(age_detail_output, age_detail.shape)
print(age_summary_output, gangnam_welfare_age_summary.shape)

저장 완료:
gangnam_general_recipient_age_detail_2024.csv (17, 2)
gangnam_general_recipient_age_summary_2024.csv (1, 11)


In [65]:
from google.colab import files

files.download(
    "gangnam_general_recipient_age_detail_2024.csv"
)

files.download(
    "gangnam_general_recipient_age_summary_2024.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>